# P10.6-AI — Notebook 63: entrenamiento subarticular Axial T2

Entrena un clasificador **2.5D multiclase** para estenosis subarticular izquierda y derecha usando exclusivamente `train_manifest.csv` y `validation_manifest.csv` del Notebook 62.

El backbone se comparte entre ambos lados e incorpora embeddings explícitos de **lado** y **nivel lumbar**. El `internal_test` permanece sellado para el Notebook 64.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


## Recursos y alcance

- Runtime recomendado: **Google Colab con GPU T4 o superior**.
- Autorizar Google Drive.
- No requiere token de GitHub.
- El token de Kaggle se solicita únicamente si las series Axial T2 necesarias no están disponibles en Drive.
- El caché local de crops `.npy` puede reutilizarse mientras no se reinicie el runtime.
- No abre ni carga `internal_test_manifest.csv`; solo verifica su existencia y el sello criptográfico.


In [ ]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util, subprocess, sys

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing
    ])
print({"installedNow": missing})


In [ ]:
# 2) GPU y Google Drive
import getpass, json, os
from pathlib import Path
import torch
from google.colab import drive  # type: ignore

if not torch.cuda.is_available():
    raise RuntimeError(
        "Seleccioná GPU T4 o superior: "
        "Entorno de ejecución > Cambiar tipo de entorno."
    )

print({
    "gpu": torch.cuda.get_device_name(0),
    "gpuMemoryGiB": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
    "torch": torch.__version__,
})
drive.mount("/content/drive", force_remount=False)


In [ ]:
# 3) Clonar o actualizar la rama e importar el pipeline
REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        ["git", "fetch", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "checkout", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "pull", "--ff-only", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

sys.path.insert(0, str(REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_subarticular_training import (
    TrainConfig,
    build_cache,
    download_selected_series,
    find_data_root,
    load_manifests,
    prepare_samples,
    train_model,
)

print({"repoRef": REPO_REF, "repoSha": REPO_SHA})


In [ ]:
# 4) Rutas y configuración
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
SPLIT_ROOT = RESULTS_ROOT / "notebook62_subarticular_split"
RUN_ROOT = RESULTS_ROOT / "notebook63_subarticular_training"

MODEL_ROOT = (
    PFI_ROOT
    / "models"
    / "P10_6_rsna_findings"
    / "subarticular_axial_t2_2p5d"
)
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"

LOCAL_DATA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
DRIVE_DATA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
CACHE_ROOT = Path("/content/rsna_subarticular_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
    minimum_macro_f1=0.36,
    minimum_balanced_accuracy=0.45,
    minimum_severe_recall=0.30,
    minimum_moderate_recall=0.25,
)

for path in (RUN_ROOT, MODEL_ROOT, CHECKPOINT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(CFG)
print({
    "splitRoot": str(SPLIT_ROOT),
    "runRoot": str(RUN_ROOT),
    "checkpointRoot": str(CHECKPOINT_ROOT),
    "cacheRoot": str(CACHE_ROOT),
})


## Carga y auditoría de datos

La siguiente celda verifica hashes, aprobación del Notebook 62, separación por `study_id`, presencia de las tres clases y el sello del internal test. Solo lee `train` y `validation`.


In [ ]:
# 5) Cargar únicamente train y validation
train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)

train_distribution = (
    train_manifest
    .groupby(["side", "level", "severity"])
    .size()
    .reset_index(name="train_rows")
)
validation_distribution = (
    validation_manifest
    .groupby(["side", "level", "severity"])
    .size()
    .reset_index(name="validation_rows")
)

print({
    "trainRows": len(train_manifest),
    "trainStudies": train_manifest["study_id"].nunique(),
    "validationRows": len(validation_manifest),
    "validationStudies": validation_manifest["study_id"].nunique(),
    "trainClassCounts": train_manifest["severity"].value_counts().to_dict(),
    "validationClassCounts": (
        validation_manifest["severity"].value_counts().to_dict()
    ),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
})
display(train_distribution)
display(validation_distribution)


In [ ]:
# 6) Resolver las series DICOM necesarias
data_root, data_audits = find_data_root(
    [LOCAL_DATA_ROOT, DRIVE_DATA_ROOT],
    train_manifest,
    validation_manifest,
)
print({
    "dataAudits": [
        {
            "root": audit.root,
            "complete": audit.complete,
            "requiredSeries": audit.required_series,
            "missingSeries": audit.missing_series,
            "missingExamples": list(audit.missing_examples),
        }
        for audit in data_audits
    ]
})

if data_root is None:
    print(
        "No se encontró un root completo. "
        "Se solicitará el token de Kaggle sin mostrarlo."
    )
    kaggle_token = getpass.getpass("Kaggle API token: " )
    data_root = download_selected_series(
        train_manifest,
        validation_manifest,
        LOCAL_DATA_ROOT,
        COMPETITION,
        kaggle_token,
    )

print({"selectedDataRoot": str(data_root)})


In [ ]:
# 7) Preparar muestras y construir caché 2.5D
train_samples = prepare_samples(
    train_manifest,
    data_root,
    "train",
)
validation_samples = prepare_samples(
    validation_manifest,
    data_root,
    "validation",
)

train_cache_audit = build_cache(
    train_samples,
    CACHE_ROOT,
    "train",
    CFG,
)
validation_cache_audit = build_cache(
    validation_samples,
    CACHE_ROOT,
    "validation",
    CFG,
)

print({
    "trainCache": train_cache_audit,
    "validationCache": validation_cache_audit,
    "internalTestAccessed": False,
})


## Entrenamiento

Se selecciona el mejor checkpoint usando exclusivamente métricas de validación. El proceso usa early stopping y no ajusta nada con el internal test.


In [ ]:
# 8) Entrenar y evaluar sobre validation
summary = train_model(
    train_samples=train_samples,
    validation_samples=validation_samples,
    cache_root=CACHE_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    run_root=RUN_ROOT,
    manifest_hashes=manifest_hashes,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    config=CFG,
)
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
# 9) Gate final y artefactos
required_outputs = [
    "training_history.csv",
    "validation_predictions.csv",
    "validation_metrics_by_group.csv",
    "sampling_audit.json",
    "model_card.md",
    "training_summary.json",
]
missing_outputs = [
    name
    for name in required_outputs
    if not (RUN_ROOT / name).is_file()
]
if missing_outputs:
    raise RuntimeError(f"Faltan outputs: {missing_outputs}")

if summary["approved"] is not True:
    raise RuntimeError(
        "El entrenamiento requiere revisión. "
        "No abrir el internal test ni ajustar gates con él."
    )
if summary["status"] != "APPROVED_FOR_NOTEBOOK_64":
    raise RuntimeError("Estado final inesperado.")
if summary["governance"]["internalTestAccessed"] is not False:
    raise RuntimeError("Se declaró acceso indebido al internal test.")

print({
    "status": summary["status"],
    "bestEpoch": summary["bestEpoch"],
    "validationMetrics": summary["validationMetrics"],
    "checkpoint": summary["checkpoint"],
    "outputs": required_outputs,
    "internalTestSealedUntilNotebook64": True,
    "officialTestAccessed": False,
})


## Resultado esperado

Una ejecución aprobada termina con `APPROVED_FOR_NOTEBOOK_64`.

El Notebook 64 abrirá una sola vez `internal_test_manifest.csv`, evaluará el checkpoint congelado y exportará el modelo final. Si algún gate falla, se conserva la evidencia de validación y no se utiliza el internal test para ajustar el modelo.
